In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from pathlib import Path
from ultralytics import YOLO
import ultralytics.data.dataset as data_dataset
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.base import BaseDataset
from ultralytics.models.yolo.detect.train import DetectionTrainer

### Tell the model that it needs to load .raw files

In [ ]:
original_get_img_files = BaseDataset.get_img_files

def custom_get_img_files(self, img_path):
    try:
        im_files = original_get_img_files(self, img_path)
    except FileNotFoundError:
        im_files = []
        
    path = Path(img_path)
    raw_files = []
    
    if path.is_dir():
        raw_files = [str(x.resolve()) for x in path.rglob("*.raw")]
    elif path.is_file() and path.suffix == ".txt":
        with open(path, "r") as f:
            raw_files = [x for x in f.read().splitlines() if x.endswith(".raw")]
            
    all_files = sorted(list(set(im_files + raw_files)))
    
    if not all_files:
        raise FileNotFoundError(f"Custom Scanner: No standard images OR .raw files found in {img_path}!")
        
    return all_files

BaseDataset.get_img_files = custom_get_img_files
print("Patched YOLO's internal file scanner to accept .raw")

### Patch the image loading for .raw files

In [ ]:
original_load_image = YOLODataset.load_image

def custom_load_image(self, i):
    im_file = self.im_files[i]

    if not im_file.endswith('.raw'):
        return original_load_image(self, i)

    with open(im_file, "rb") as f:
        raw_data = f.read()

        width, height = 640, 480
    rgb_bytes_len = width * height * 3

    rgb = np.frombuffer(raw_data[:rgb_bytes_len], dtype=np.uint8).reshape(
        (height, width, 3)
    )
    depth = np.frombuffer(raw_data[rgb_bytes_len:], dtype=np.uint16).reshape(
        (height, width, 1)
    )

    depth_norm = (depth / depth.max() * 255).astype(np.uint8)

    im = np.concatenate([rgb, depth_norm], axis=-1)
    return im, (height, width), im.shape[:2]


YOLODataset.load_image = custom_load_image

### Patch Pillow to load .raw files

In [ ]:
original_pil_open = Image.open

def custom_pil_open(fp, mode='r', formats=None):
    if str(fp).endswith('.raw'):
        
        dummy_img = Image.new('RGB', (640, 480))
        dummy_img.verify = lambda: None 
        
        return dummy_img
    
    return original_pil_open(fp, mode=mode, formats=formats)

if not hasattr(Image, '_is_patched_for_raw'):
    Image.open = custom_pil_open
    Image._is_patched_for_raw = True
    print("Patched PIL to bypass YOLO image verification!")
else:
    print("PIL is already patched.")

### Create custom label verification, ensure that all labels are bounding boxes
this is a scary one

In [ ]:
import numpy as np
from pathlib import Path

original_verify = data_dataset.verify_image_label 

def custom_verify(args):
    im_file, lb_file = args[0], args[1]
    
    # Ignore non-raw files
    if not str(im_file).endswith('.raw'):
        return original_verify(args)
        
    shape = (480, 640)
    segments, keypoints = [], None
    empty_label = np.zeros((0, 5), dtype=np.float32)
    
    lb_path = Path(lb_file)
    
    # 3. Handle missing labels
    if not lb_path.exists():
        return im_file, empty_label, shape, segments, keypoints, 1, 0, 0, 0, 'missing'
            
    try:
        with lb_path.open('r') as f:
            lines = [x.strip().split() for x in f.read().splitlines() if x.strip()]
            
        # Handle empty labels
        if not lines:
            return im_file, empty_label, shape, segments, keypoints, 0, 0, 1, 0, 'empty'
            
        boxes = []
        for line in lines:
            cls = float(line[0])
            coords = [float(x) for x in line[1:]]
            
            if len(coords) == 4:
                boxes.append([cls] + coords)
            elif len(coords) > 4 and len(coords) % 2 == 0:
                # Convert Polygon to Bounding Box
                xs, ys = coords[0::2], coords[1::2]
                min_x, max_x = min(xs), max(xs)
                min_y, max_y = min(ys), max(ys)
                
                cx, cy = (min_x + max_x) / 2, (min_y + max_y) / 2
                w, h = max_x - min_x, max_y - min_y
                boxes.append([cls, cx, cy, w, h])
                    
        lb = np.array(boxes, dtype=np.float32)
        return im_file, lb, shape, segments, keypoints, 0, 1, 0, 0, ''
                
    except Exception as e:
        print(f"Corrupt label file {lb_file}: {e}")
        return im_file, empty_label, shape, segments, keypoints, 0, 0, 0, 1, str(e)

### Validate dataset

In [ ]:
train_img_dir = "annotated/train/images"
train_lbl_dir = "annotated/train/labels"

raw_files = [Path(f).stem for f in glob.glob(f"{train_img_dir}/*.raw")]
print(f"Found {len(raw_files)} .raw images.")

txt_files = glob.glob(f"{train_lbl_dir}/*.txt")
print(f"Found {len(txt_files)} .txt labels.")

renamed_count = 0
for txt_path in txt_files:
    txt_name = Path(txt_path).stem
    
    if "_png.rf." in txt_name or ".rf." in txt_name:
        clean_name = txt_name.split("_png.rf.")[0].split(".rf.")[0]
        
        new_path = os.path.join(train_lbl_dir, f"{clean_name}.txt")
        os.rename(txt_path, new_path)
        renamed_count += 1

if renamed_count > 0:
    print(f"Renamed {renamed_count} labels to remove Roboflow hashes!")

fixed_txt_files = [Path(f).stem for f in glob.glob(f"{train_lbl_dir}/*.txt")]
matches = set(raw_files).intersection(set(fixed_txt_files))
print(f"Paired {len(matches)} images with labels.")

if len(matches) == 0:
    print("There are still no matching filenames between your images and labels!")
    print("Example image name:", raw_files[0] if raw_files else "None")
    print("Example label name:", fixed_txt_files[0] if fixed_txt_files else "None")

### Patch the default image verification function

In [ ]:
if not hasattr(data_dataset, '_verify_patched'):
    data_dataset.verify_image_label = custom_verify
    data_dataset._verify_patched = True
    print("Patched YOLO's dataset verification namespace (with Polygon-to-BBox conversion)!")
else:
    print("Label verification already patched!")

### Clean cache

In [ ]:
cache_files = glob.glob('annotated/**/*.cache', recursive=True)
if cache_files:
    for f in cache_files:
        os.remove(f)
        print(f"Deleted corrupt cache: {f}")
else:
    print("No caches found. Clean slate!")

### Define a 4-layered YOLO model

In [ ]:
original_get_model = DetectionTrainer.get_model

def custom_get_model(self, cfg=None, weights=None, verbose=True):
        model = original_get_model(self, cfg, weights, verbose)
        
        first_module = model.model[0]
        
        if first_module.conv.in_channels == 3:
            old_conv = first_module.conv
            
            new_conv = nn.Conv2d(
                in_channels=4, 
                out_channels=old_conv.out_channels, 
                kernel_size=old_conv.kernel_size, 
                stride=old_conv.stride, 
                padding=old_conv.padding, 
                bias=(old_conv.bias is not None)
            )
            
            new_conv.weight.data[:, :3, :, :] = old_conv.weight.data
            new_conv.weight.data[:, 3:4, :, :] = old_conv.weight.data.mean(dim=1, keepdim=True)
            
            if old_conv.bias is not None:
                new_conv.bias.data = old_conv.bias.data
                
            first_module.conv = new_conv
            print("Intercepted Trainer: Re-applied 4-channel conversion for training!")
            
        return model

### Patch patch patch!!

In [ ]:
if not hasattr(DetectionTrainer, '_get_model_patched'):
    DetectionTrainer.get_model = custom_get_model
    DetectionTrainer._get_model_patched = True
    print("Patched YOLO Trainer to support 4-channel models!")
else:
    print("YOLO Trainer already patched!")

### Start training

In [ ]:
yaml_path = "annotated/data.yaml"

print("Loading base YOLO model...")
model = YOLO('yolo11n.pt') 

print(f"Starting training on dataset in {yaml_path}...")
results = model.train(
    data=yaml_path,
    epochs=50,       # Adjust based on your needs
    imgsz=640,       # Matches the dimensions in your loader
    batch=16,        # Adjust based on your GPU VRAM
    workers=0,       # Forces data loading to happen in this main thread
    device='cpu',        # Use 0 for GPU, or 'cpu' if no GPU is available
    project='rgbd_yolo', # Directory to save training runs
    name='run1',
    mosaic=0.0,  # Disables the augmentation causing the buffer crash
    mixup=0.0    # Disables another multi-image augment that assumes 3 channels
)

print("Training complete!")